# 16.2 BM25 & probabilistic retrieval

BM25 ranks keyword matches by rewarding rare query terms, saturating repeated terms, and correcting for document length.

BM25 is what happens when the exact candidate machinery of 16.1 grows a careful ranking function. Inverted indexes provide tf and df without scanning every document. Logarithms turn rarity into a stable additive weight, while length correction keeps long documents from winning by accident. Save a copy to Drive to edit.

In [ ]:

import math
import re
import signal
import random
from collections import Counter
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

SEED = 16
random.seed(SEED)
np.random.seed(SEED)


def tokenize(text, lowercase=True, synonym_map=None, keep_case=False):
    if lowercase and not keep_case:
        text = text.lower()
    tokens = re.findall(r"[A-Za-z0-9_'-]+", text)
    if synonym_map is None:
        return tokens
    mapped = []
    for token in tokens:
        key = token.lower()
        mapped.append(synonym_map.get(key, key))
    return mapped


def ranked_doc_ids(scores, reverse=True):
    return [doc_id for doc_id, score in sorted(scores.items(), key=lambda item: (-item[1], item[0]) if reverse else (item[1], item[0]))]


def recall_at_k(ranking, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    hits = len(set(ranking[:k]) & relevant)
    return hits / len(relevant)


def dcg_at_k(ranking, gains, k):
    total = 0.0
    for rank, doc_id in enumerate(ranking[:k], start=1):
        gain = gains.get(doc_id, 0.0)
        total += gain / math.log2(rank + 1)
    return total


def ndcg_at_k(ranking, gains, k):
    ideal = sorted(gains.values(), reverse=True)
    ideal_total = 0.0
    for rank, gain in enumerate(ideal[:k], start=1):
        ideal_total += gain / math.log2(rank + 1)
    if ideal_total == 0:
        return 0.0
    return dcg_at_k(ranking, gains, k) / ideal_total


def mrr(ranking, relevant):
    relevant = set(relevant)
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def preview_rungs(rungs):
    for rung in rungs:
        docs = rung["docs"]
        print(rung["name"], "docs=", len(docs), "query=", rung["query"])
        sample_ids = list(docs)[:3]
        for doc_id in sample_ids:
            print(" ", doc_id, "->", docs[doc_id][:90])
        print()


def try_fetch_20newsgroups_subset(categories, max_docs=18, timeout_seconds=3):
    def timeout_handler(signum, frame):
        raise TimeoutError("fetch_20newsgroups timed out")
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)
    try:
        from sklearn.datasets import fetch_20newsgroups
        data = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
        texts = [text.replace("\n", " ") for text in data.data[:max_docs]]
        return texts
    except Exception:
        return None
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


def inline_newsgroups_docs():
    return {
        1: "space shuttle orbit nasa launch telescope mission",
        2: "hockey puck goalie playoff ice team",
        3: "graphics rendering image pixel shader gpu",
        4: "nasa mars orbit satellite telemetry",
        5: "goalie blocks puck in hockey playoff",
        6: "image compression rendering color pixel",
        7: "space telescope observes galaxy orbit",
        8: "team wins ice hockey tournament",
        9: "gpu shader renders three dimensional graphics",
        10: "satellite launch vehicle reaches orbit",
        11: "playoff team trades veteran goalie",
        12: "graphics image pipeline uses antialiasing",
    }


def lesson_docs_boolean():
    return {
        1: "neural search search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search search",
    }


def lesson_docs_sets():
    return {
        1: "neural search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search",
    }


def retrieval_ladder(kind):
    if kind == "boolean":
        return [
            {
                "name": "D1 lesson toy corpus",
                "docs": lesson_docs_sets(),
                "query": "search AND graph",
                "relevant": {2, 4},
            },
            {
                "name": "D2 12 clean topic docs",
                "docs": {
                    1: "neural search ranking",
                    2: "graph retrieval index",
                    3: "recipe ingredient oven",
                    4: "neural embeddings search",
                    5: "legal discovery contract",
                    6: "graph database traversal",
                    7: "search engine crawler",
                    8: "retrieval evaluation recall",
                    9: "support ticket reset",
                    10: "neural network training",
                    11: "index postings compression",
                    12: "semantic search retrieval",
                },
                "query": "neural AND search",
                "relevant": {1, 4},
            },
            {
                "name": "D3 case synonyms ties",
                "docs": {
                    1: "Search platform handles neural queries",
                    2: "semantic matching helps search",
                    3: "NEURAL retrieval benchmark",
                    4: "graph search retrieval",
                    5: "support SEARCH portal",
                    6: "neural Search ranking",
                    7: "lexical lookup only",
                    8: "semantic retrieval without keyword",
                    9: "search neural tie case",
                    10: "faq finder synonym lookup",
                    11: "audit log exclusion filter",
                    12: "rareterm alpha beta",
                },
                "query": "neural AND search",
                "relevant": {1, 6, 9},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space AND orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail negation rare terms",
                "docs": {
                    1: "urgent security incident xj9 raretoken search neural allowlist",
                    2: "security incident xj9 raretoken malware neural",
                    3: "Search incident raretoken benign customer support",
                    4: "security search xj9 raretoken not neural",
                    5: "compliance archive common logs",
                    6: "raretoken investigation search neural case study",
                    7: "neural search raretoken exclude deprecated",
                    8: "xj9 raretoken Search security search",
                    9: "incident response without keyword",
                    10: "rareterm unrelated graph retrieval",
                    11: "security neural search raretoken",
                    12: "deprecated neural search raretoken",
                },
                "query": "raretoken AND search AND NOT deprecated",
                "relevant": {1, 3, 4, 6, 8, 11},
            },
        ]
    if kind == "bm25":
        return [
            {
                "name": "D1 lesson BM25 corpus",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "gains": {1: 3, 3: 2, 4: 1, 2: 0},
            },
            {
                "name": "D2 clean keyword docs",
                "docs": {
                    1: "neural search ranking relevance",
                    2: "search ranking index",
                    3: "neural retrieval embeddings",
                    4: "graph database traversal",
                    5: "support search help center",
                    6: "neural network optimizer",
                    7: "enterprise search neural",
                    8: "retrieval evaluation ndcg",
                    9: "legal discovery query",
                    10: "search logs observability",
                    11: "neural search relevance",
                    12: "lexical index postings",
                },
                "query": "neural search",
                "gains": {1: 3, 7: 3, 11: 3, 3: 1, 6: 1, 2: 1, 5: 1, 10: 1},
            },
            {
                "name": "D3 stuffed repeats length variation",
                "docs": {
                    1: "neural search relevance",
                    2: "search search search search search coupon",
                    3: "neural retrieval scientific paper",
                    4: "search neural search neural ranking",
                    5: "very long document with search filler filler filler filler filler",
                    6: "graph traversal retrieval",
                    7: "neural semantic search",
                    8: "search support support support",
                    9: "neural neural model only",
                    10: "ranking index postings",
                    11: "search neural anti stuffing",
                    12: "irrelevant repeated repeated repeated",
                },
                "query": "neural search",
                "gains": {1: 3, 4: 3, 7: 3, 11: 3, 3: 1, 9: 1, 2: 0, 5: 0},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "gains": {1: 3, 4: 3, 7: 3, 10: 3},
            },
            {
                "name": "D5 long docs synonym misses",
                "docs": {
                    1: "neural search ranking tutorial with exact query terms",
                    2: "semantic retrieval finds meaning with embeddings and vectors",
                    3: "enterprise findability platform with semantic matching but no keyword overlap",
                    4: "neural neural neural search search stuffed landing page",
                    5: "long audit report search appendix appendix appendix appendix appendix appendix",
                    6: "support answer retrieval help article",
                    7: "semantic vector matching for neural information need",
                    8: "search logs with unrelated navigation data",
                    9: "dense encoder synonym recall for faq finder",
                    10: "legal discovery keyword exact match",
                    11: "neural search production benchmark",
                    12: "meaning based lookup with embeddings",
                },
                "query": "neural search",
                "gains": {1: 3, 2: 2, 3: 2, 7: 2, 9: 2, 11: 3, 4: 1},
            },
        ]
    if kind == "tfidf":
        return [
            {
                "name": "D1 lesson sparse vectors",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "relevant": {1, 3, 4},
            },
            {
                "name": "D2 clean TF-IDF mini corpus",
                "docs": {
                    1: "neural search ranking",
                    2: "search index postings",
                    3: "neural embeddings retrieval",
                    4: "graph database",
                    5: "semantic search retrieval",
                    6: "support center",
                    7: "neural search evaluation",
                    8: "legal discovery",
                    9: "query expansion synonym",
                    10: "search logs",
                    11: "neural model",
                    12: "ranking relevance search",
                },
                "query": "neural search",
                "relevant": {1, 7, 3, 5, 11},
            },
            {
                "name": "D3 stopwords synonyms ties",
                "docs": {
                    1: "the neural search system",
                    2: "the car repair guide",
                    3: "automobile maintenance manual",
                    4: "search and the retrieval",
                    5: "neural retrieval and search",
                    6: "vehicle diagnostics handbook",
                    7: "the the the search",
                    8: "semantic lookup finder",
                    9: "neural model guide",
                    10: "the and of support",
                    11: "search neural duplicate",
                    12: "graph retrieval index",
                },
                "query": "neural search",
                "relevant": {1, 5, 9, 11},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail vocab stopword-heavy docs",
                "docs": {
                    1: "car insurance claim policy vehicle accident",
                    2: "automobile coverage deductible crash report",
                    3: "the the the and of to in car",
                    4: "vehicle repair parts estimate",
                    5: "neural search unrelated tutorial",
                    6: "automobile automobile warranty service",
                    7: "car rental booking airport",
                    8: "rare sku xj9 part compatibility vehicle",
                    9: "support article about login reset",
                    10: "auto policy renewal notice",
                    11: "vehicle crash legal claim",
                    12: "long stopword document the and of in to with for on by",
                },
                "query": "car accident claim",
                "relevant": {1, 2, 4, 8, 10, 11},
            },
        ]
    if kind == "dense":
        return [
            {
                "name": "D1 lesson 2D embeddings",
                "docs": {
                    1: "semantic search match",
                    2: "faq synonym neighbor",
                    3: "orthogonal mismatch",
                    4: "opposite constraint",
                },
                "query": "semantic search",
                "embeddings": np.array([[1.0, 0.0], [0.8, 0.2], [0.0, 1.0], [-0.2, 0.9]]),
                "query_vector": np.array([0.9, 0.1]),
                "relevant": {1, 2},
            },
            {
                "name": "D2 clean semantic clusters",
                "docs": {i + 1: text for i, text in enumerate([
                    "password reset login help",
                    "account sign in recovery",
                    "invoice billing payment",
                    "credit card receipt",
                    "model training embeddings",
                    "vector retrieval semantic",
                    "shipping delivery package",
                    "order tracking courier",
                    "legal contract review",
                    "compliance policy audit",
                    "search ranking relevance",
                    "faq answer support",
                ])},
                "query": "login recovery",
                "relevant": {1, 2, 12},
            },
            {
                "name": "D3 synonyms noisy high-norm vectors",
                "docs": {i + 1: text for i, text in enumerate([
                    "car repair manual",
                    "automobile service guide",
                    "vehicle maintenance tips",
                    "billing payment invoice",
                    "login account help",
                    "huge norm noisy unrelated",
                    "auto insurance claim",
                    "engine diagnostic checklist",
                    "travel hotel booking",
                    "graph neural search",
                    "support ticket reset",
                    "semantic finder retrieval",
                ])},
                "query": "car maintenance",
                "relevant": {1, 2, 3, 7, 8},
            },
            {
                "name": "D4 20newsgroups tiny hashed SVD embeddings",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 off-domain constraint-heavy queries",
                "docs": {i + 1: text for i, text in enumerate([
                    "refund policy for enterprise plan excluding trials",
                    "trial cancellation guide no refund guaranteed",
                    "enterprise contract renewal refund clause",
                    "sports hockey playoff recap",
                    "graphics gpu shader discussion",
                    "refund exception for medical emergency",
                    "login reset support article",
                    "enterprise billing dispute with legal constraint",
                    "semantic neighbor about repayment but missing enterprise",
                    "off domain recipe ingredients",
                    "legal contract refund negotiation",
                    "high norm generic vector placeholder",
                ])},
                "query": "enterprise refund not trial",
                "relevant": {1, 3, 8, 11},
            },
        ]
    raise ValueError(kind)


def bm25_index(docs):
    tokenized = {doc_id: tokenize(text) for doc_id, text in docs.items()}
    lengths = {doc_id: len(tokens) for doc_id, tokens in tokenized.items()}
    avgdl = sum(lengths.values()) / len(lengths)
    df = Counter()
    for tokens in tokenized.values():
        for token in set(tokens):
            df[token] += 1
    return tokenized, lengths, avgdl, df


def bm25_idf(term, df, n_docs):
    value = math.log(1.0 + (n_docs - df.get(term, 0) + 0.5) / (df.get(term, 0) + 0.5))
    return value


def bm25_score(query, docs, k1=1.2, b=0.75):
    tokenized, lengths, avgdl, df = bm25_index(docs)
    query_terms = tokenize(query)
    scores = {}
    contributions = {}
    for doc_id, tokens in tokenized.items():
        counts = Counter(tokens)
        score = 0.0
        doc_contribs = {}
        for term in query_terms:
            tf = counts.get(term, 0)
            if tf == 0:
                doc_contribs[term] = 0.0
                continue
            idf = bm25_idf(term, df, len(docs))
            denom = tf + k1 * (1.0 - b + b * lengths[doc_id] / avgdl)
            contrib = idf * (tf * (k1 + 1.0)) / denom
            score += contrib
            doc_contribs[term] = contrib
        scores[doc_id] = score
        contributions[doc_id] = doc_contribs
    return scores, contributions, df, avgdl


## The concept, built once (D1)

BM25 scores $$	ext{BM25}(q,d)=\sum_{t\in q} idf(t)rac{tf_{t,d}(k_1+1)}{tf_{t,d}+k_1(1-b+b|d|/avgdl)}$$ with $idf(t)=\log\left(1+rac{N-df_t+0.5}{df_t+0.5}ight)$, $k_1=1.2$, and $b=0.75$. The lesson corpus has lengths $3,2,3,4$, so $avgdl=3.000$.

In [ ]:

docs = lesson_docs_boolean()
scores, contributions, df, avgdl = bm25_score("neural search", docs)
search_idf = bm25_idf("search", df, len(docs))
neural_idf = bm25_idf("neural", df, len(docs))

assert round(avgdl, 3) == 3.000
assert round(search_idf, 3) == 0.357
assert round(neural_idf, 3) == 0.693
assert round(scores[1], 3) == 1.184

print("avgdl", round(avgdl, 3))
print("idf(search)", round(search_idf, 3))
print("idf(neural)", round(neural_idf, 3))
print("d1 score", round(scores[1], 3))


The exact D1 ranking should be $d_1>d_3>d_4>d_2$: one rare `neural` match in $d_3$ ($0.693$) beats two common-only `search` matches in $d_4$ ($0.448$), while $d_2$ has only one common term ($0.413$).

In [ ]:

ranking = ranked_doc_ids(scores)

assert ranking == [1, 3, 4, 2]
assert round(scores[3], 3) == 0.693
assert round(scores[4], 3) == 0.448
assert round(scores[2], 3) == 0.413

for doc_id in ranking:
    print(doc_id, round(scores[doc_id], 3), contributions[doc_id])


## The dataset ladder

D1 verifies lesson numbers, D2 is a clean keyword corpus, D3 adds stuffing and length variation, D4 uses a small 20newsgroups-style inline fallback, and D5 stresses long documents plus synonym-only relevant documents BM25 cannot see.

In [ ]:

rungs = retrieval_ladder("bm25")
preview_rungs(rungs)


In [ ]:

rows = []
artifacts = []
for rung in rungs:
    scores, contributions, df, avgdl = bm25_score(rung["query"], rung["docs"])
    ranking = ranked_doc_ids(scores)
    metric = ndcg_at_k(ranking, rung["gains"], 5)
    rows.append((rung["name"], metric, ranking[:5]))
    artifacts.append((rung, scores, contributions, ranking))

print("rung | NDCG@5 | top5")
for name, metric, top5 in rows:
    print(f"{name} | {metric:.3f} | {top5}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, (rung, scores, contributions, ranking) in enumerate(artifacts):
    top = ranking[:5]
    axes[0, col].bar([str(doc_id) for doc_id in top], [scores[doc_id] for doc_id in top])
    axes[0, col].set_title(rung["name"].split()[0])
    axes[0, col].set_ylabel("BM25")
    query_terms = tokenize(rung["query"])
    bottom = np.zeros(len(top))
    for term in query_terms:
        values = [contributions[doc_id].get(term, 0.0) for doc_id in top]
        axes[1, col].bar([str(doc_id) for doc_id in top], values, bottom=bottom, label=term)
        bottom = bottom + np.array(values)
    axes[1, col].set_title("term contributions")

metric_values = [row[1] for row in rows]
fig2, ax = plt.subplots(figsize=(6, 3))
ax.plot(["D1", "D2", "D3", "D4", "D5"], metric_values, marker="o")
ax.set_ylim(0, 1.05)
ax.set_ylabel("NDCG@5")
ax.set_title("NDCG@5 vs complexity")
plt.show()


## Pitfall on D5: expecting semantic recall

BM25 cannot score synonym-only relevant documents when none of the query terms overlap. A small hybrid note maps synonyms to the query vocabulary so those documents can at least enter the candidate set.

In [ ]:

d5 = rungs[-1]
base_scores, base_contribs, base_df, base_avgdl = bm25_score(d5["query"], d5["docs"])
base_ranking = ranked_doc_ids(base_scores)
base_metric = ndcg_at_k(base_ranking, d5["gains"], 5)
synonyms = {
    "semantic": "neural",
    "embeddings": "neural",
    "encoder": "neural",
    "findability": "search",
    "lookup": "search",
    "finder": "search",
}
expanded_docs = {doc_id: " ".join(tokenize(text, synonym_map=synonyms)) for doc_id, text in d5["docs"].items()}
expanded_scores, expanded_contribs, expanded_df, expanded_avgdl = bm25_score(d5["query"], expanded_docs)
expanded_ranking = ranked_doc_ids(expanded_scores)
expanded_metric = ndcg_at_k(expanded_ranking, d5["gains"], 5)

assert expanded_metric >= base_metric
print("base top5", base_ranking[:5], "NDCG@5", round(base_metric, 3))
print("expanded top5", expanded_ranking[:5], "NDCG@5", round(expanded_metric, 3))


## Evaluate it + Practice

- Report the planned metric (NDCG@5) beside a no-skill baseline such as random ranking or first-k document order.
- Run a cheap sanity check: the D1 asserts should pass before trusting any harder rung.
- Ablate the key idea and expect the metric to drop: set idf to 1 or b to 0 and observe stuffed or long documents rise.
- Watch failure signals: empty candidates, tied scores, case splits, synonym misses, and high-norm artifacts.
- Keep all examples CPU-only, seeded, and small enough for a notebook cell.

Practice prompts:
1. Change one relevance label on D3 and recompute the metric table.



2. Change $k_1$ from 1.2 to 2.0 and inspect term-frequency saturation.

3. Remove the synonym expansion on D5 and compare the missed relevant documents.